In [1]:
import os
import re
import io
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
from nltk.corpus import stopwords
import pickle

def clean_text(text):
    """
    Clean text by removing unwanted characters and formatting.
    """
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII characters
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters and numbers
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    text = re.sub(r'\n+', '\n', text)  # Remove extra newlines
    return text


def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)


def preprocess_pdf(file_content, filename):
    """Extract and preprocess text from uploaded PDF file."""
    try:
        # Convert memoryview to BytesIO
        pdf_stream = io.BytesIO(file_content)

        reader = PdfReader(pdf_stream)
        extracted_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                extracted_text += text + "\n"

        cleaned_text = clean_text(extracted_text)
        cleaned_text = remove_stop_words(cleaned_text)

        # Save cleaned text to file
        output_path = os.path.join(os.getcwd(), f"{filename}.txt")
        with open(output_path, "w", encoding="utf-8") as text_file:
            text_file.write(cleaned_text)

        print(f"Processed text saved to: {output_path}")
        print(f"Preview:\n{cleaned_text[:500]}...")  # Print first 500 chars

        # Save metadata for second script
        metadata = {
            "filename": filename,
            "file_path": output_path
        }
        with open("processed_file.pkl", "wb") as f:
            pickle.dump(metadata, f)

        print("Metadata saved for evaluation script.")

    except Exception as e:
        print(f"Error processing PDF: {e}")


# Upload Widget
upload_widget = widgets.FileUpload(
    accept='.pdf',  # Accept only PDF files
    multiple=False  # Only allow single file upload
)


def on_upload_change(change):
    """Handle file upload and process PDF."""
    if upload_widget.value:
        for file_info in upload_widget.value:
            filename = file_info['name'].replace(".pdf", "")
            file_content = file_info['content'].tobytes()  # Convert memoryview to bytes
            
            # Process PDF
            preprocess_pdf(file_content, filename)


# Attach event listener
upload_widget.observe(on_upload_change, names='value')

display(upload_widget)

FileUpload(value=(), accept='.pdf', description='Upload')

In [8]:
for file_info in upload_widget.value:
    print(file_info.name)

pillar3-disclosures-1q-2018.pdf


In [2]:
print(upload_widget.value)

({'name': 'pillar3-disclosures-1q-2018.pdf', 'type': 'application/pdf', 'size': 98305, 'content': <memory at 0x15230fac0>, 'last_modified': datetime.datetime(2025, 3, 1, 7, 48, 13, 135000, tzinfo=datetime.timezone.utc)},)


In [37]:
import os
import pickle
import json
import pandas as pd
import boto3
from botocore.config import Config
from dotenv import load_dotenv

load_dotenv("codes.env")

# AWS credentials
aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
aws_region = os.environ.get("AWS_REGION")

# AWS Bedrock model configuration
MODEL_ID_LLAMA = "arn:aws:bedrock:us-west-2:874280117166:inference-profile/us.meta.llama3-3-70b-instruct-v1:0"

# Prevent Bedrock timeout
config = Config(read_timeout=1000)

client = boto3.client(
    "bedrock-runtime",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    config=config
)

# Load topic mappings
mapping_file_path = 'final_file_topic_mapping.csv'
file_topic_mapping = pd.read_csv(mapping_file_path)
unique_topics = file_topic_mapping['folder_name'].unique().tolist()
unique_topics_str = ', '.join(unique_topics)


def read_txt_file(file_path):
    """Reads the content of a .txt file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None


def map_to_category(predicted_output):
    """Maps the model's output to a known category."""
    predicted_output = predicted_output.lower().strip()
    for topic in unique_topics:
        if topic.lower() in predicted_output:
            return topic
    return "unknown"


def evaluate_topic_with_llama(file_content):
    """Classify the text using AWS Bedrock (Meta's Llama 3.3 70B Instruct)."""
    try:
        prompt = f"Classify the following text into only one of these topics: {unique_topics_str}. \n{file_content}"
        formatted_prompt = f"""
            <|begin_of_text|>
            <|start_header_id|>user<|end_header_id|>
            {prompt}
            <|eot_id|>
            <|start_header_id|>assistant<|end_header_id|>
            """

        response = client.invoke_model(
            modelId=MODEL_ID_LLAMA,
            body=json.dumps({
                "prompt": formatted_prompt,
                "max_gen_len": 512,
                "temperature": 0,
            }),
            contentType="application/json"
        )
        response_body = json.loads(response['body'].read())
        predicted_topic = response_body.get("generation", "").strip()
        
        if not predicted_topic:
            print("Empty response from AWS Bedrock Llama, defaulting to unknown.")

        return map_to_category(predicted_topic)

    except Exception as e:
        print(f"Error calling AWS Bedrock API: {e}")
        return "unknown"


def evaluate_saved_file():
    """Loads metadata, reads file content, and evaluates it."""
    try:
        # Load metadata
        with open("processed_file.pkl", "rb") as f:
            metadata = pickle.load(f)

        filename = metadata["filename"]
        file_path = metadata["file_path"]

        print(f"Evaluating file: {filename}")

        # Read file content
        text_content = read_txt_file(file_path)
        if text_content:
            predicted_topic = evaluate_topic_with_llama(text_content)
            print(f"Predicted Topic: {predicted_topic}")
        else:
            print("Error: No content found in the file.")

    except FileNotFoundError:
        print("Error: No processed file metadata found. Run `upload_pdf.py` first.")
    except Exception as e:
        print(f"Unexpected error: {e}")


# Run the evaluation
evaluate_saved_file()

Evaluating file: 626 Banks_GCO vetted
Predicted Topic: Anti Money Laundering


# showing demo of explanation before predicted topic:

In [3]:
import os
import pickle
import json
import pandas as pd
import boto3
import re
from botocore.config import Config
from dotenv import load_dotenv

load_dotenv("codes.env")

# AWS credentials
aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
aws_region = os.environ.get("AWS_REGION")

# AWS Bedrock model configuration
MODEL_ID_LLAMA = "us.meta.llama3-3-70b-instruct-v1:0"

# Prevent Bedrock timeout
config = Config(read_timeout=1000)

client = boto3.client(
    "bedrock-runtime",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    config=config
)

# Load topic mappings
mapping_file_path = 'final_file_topic_mapping.csv'
file_topic_mapping = pd.read_csv(mapping_file_path)
unique_topics = file_topic_mapping['folder_name'].unique().tolist()
unique_topics_str = ', '.join(unique_topics)

def read_txt_file(file_path):
    """Reads the content of a .txt file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            return file.read()
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return None

def extract_final_topic(response_text):
    """
    Extracts the final topic from the model's response using regex.
    Ensures that we capture the topic stated explicitly at the end.
    """
    match = re.search(r"Final Topic:\s*(.+)", response_text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    
    # If no clear label is found, fall back to the last line
    lines = response_text.strip().split("\n")
    return lines[-1].strip() if lines else "unknown"

def evaluate_topic_with_llama(file_content):
    """Classify the text using AWS Bedrock (Meta's Llama 3.3 70B Instruct)."""
    try:
        prompt = f"Classify the following text into only one of these topics: {unique_topics_str}. \n{file_content}"
        formatted_prompt = f"""
            <|begin_of_text|>
            <|start_header_id|>user<|end_header_id|>
            {prompt}
            <|eot_id|>
            <|start_header_id|>assistant<|end_header_id|>
            """

        response = client.invoke_model(
            modelId=MODEL_ID_LLAMA,
            body=json.dumps({
                "prompt": formatted_prompt,
                "max_gen_len": 512,
                "temperature": 0,
            }),
            contentType="application/json"
        )
        response_body = json.loads(response['body'].read())
        predicted_topic = response_body.get("generation", "").strip()
        
        if not predicted_topic:
            print("Empty response from AWS Bedrock Llama, defaulting to unknown.")

        return map_to_category(predicted_topic)

    except Exception as e:
        print(f"Error calling AWS Bedrock API: {e}")
        return "unknown"

def evaluate_saved_file():
    """Loads metadata, reads file content, and evaluates it with explanation."""
    try:
        # Load metadata
        with open("processed_file.pkl", "rb") as f:
            metadata = pickle.load(f)

        filename = metadata["filename"]
        file_path = metadata["file_path"]

        print(f"Evaluating file: {filename}")

        # Read file content
        text_content = read_txt_file(file_path)
        if text_content:
            explanation, predicted_topic = evaluate_topic_with_llama(text_content)
            print(f"Explanation: {explanation}\nPredicted Topic: {predicted_topic}")
        else:
            print("Error: No content found in the file.")

    except FileNotFoundError:
        print("Error: No processed file metadata found. Run `upload_pdf.py` first.")
    except Exception as e:
        print(f"Unexpected error: {e}")

# Run the evaluation

evaluate_saved_file()

Evaluating file: pillar3-disclosures-1q-2018
Error calling AWS Bedrock API: name 'map_to_category' is not defined
Unexpected error: too many values to unpack (expected 2)


# Demo with Rule Based Classification

In [ ]:
import os
import re
import io
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
from nltk.corpus import stopwords
import pickle

def clean_text(text):
    """
    Clean text by removing unwanted characters and formatting.
    """
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII characters
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters and numbers
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    text = re.sub(r'\n+', '\n', text)  # Remove extra newlines
    return text


def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)


def preprocess_pdf(file_content, filename):
    """Extract and preprocess text from uploaded PDF file."""
    try:
        # Convert memoryview to BytesIO
        pdf_stream = io.BytesIO(file_content)

        reader = PdfReader(pdf_stream)
        extracted_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                extracted_text += text + "\n"

        cleaned_text = clean_text(extracted_text)
        cleaned_text = remove_stop_words(cleaned_text)

        # Save cleaned text to file
        output_path = os.path.join(os.getcwd(), f"{filename}.txt")
        with open(output_path, "w", encoding="utf-8") as text_file:
            text_file.write(cleaned_text)

        print(f"Processed text saved to: {output_path}")
        print(f"Preview:\n{cleaned_text[:500]}...")  # Print first 500 chars

        # Save metadata for second script
        metadata = {
            "filename": filename,
            "file_path": output_path
        }
        with open("processed_file.pkl", "wb") as f:
            pickle.dump(metadata, f)

        print("Metadata saved for evaluation script.")

    except Exception as e:
        print(f"Error processing PDF: {e}")


# Upload Widget
upload_widget = widgets.FileUpload(
    accept='.pdf',  # Accept only PDF files
    multiple=False  # Only allow single file upload
)


def on_upload_change(change):
    """Handle file upload and process PDF."""
    if upload_widget.value:
        for file_info in upload_widget.value:
            filename = file_info['name'].replace(".pdf", "")
            file_content = file_info['content'].tobytes()  # Convert memoryview to bytes
            
            # Process PDF
            preprocess_pdf(file_content, filename)


# Attach event listener
upload_widget.observe(on_upload_change, names='value')

display(upload_widget)

In [13]:
print(upload_widget.value)

({'name': '626 Banks_GCO vetted.pdf', 'type': 'application/pdf', 'size': 166662, 'content': <memory at 0x17a5ca680>, 'last_modified': datetime.datetime(2025, 3, 1, 7, 10, 57, 155000, tzinfo=datetime.timezone.utc)},)


In [20]:
import os
import pickle
import json
import pandas as pd
import math
import re
from sklearn.feature_extraction.text import CountVectorizer
import boto3
from botocore.config import Config
from dotenv import load_dotenv

#  Load Environment Variables
load_dotenv("codes.env")

aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
aws_region = os.environ.get("AWS_REGION")

MODEL_ID_LLAMA = "us.meta.llama3-3-70b-instruct-v1:0"
config = Config(read_timeout=1000)

bedrock_client = boto3.client(
    "bedrock-runtime",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    config=config
)

#  Load Predefined Topics & Mappings
mapping_file_path = 'final_file_topic_mapping.csv'
file_topic_mapping = pd.read_csv(mapping_file_path)
unique_topics = file_topic_mapping['folder_name'].unique().tolist()
unique_topics_str = ', '.join(unique_topics)

#  Load Keyword Data for Rule-Based Classification
top_keywords_per_topic = {}
main_topics = set()

df = pd.read_csv("refined_tfidf_bigrams.csv")
for index, row in df.iterrows():
    topic_name = row.iloc[0].strip()
    keywords = [row[col] for col in df.columns[1:] if pd.notna(row[col])]
    if keywords:
        top_keywords_per_topic[topic_name] = keywords[:150]
        main_topics.add(topic_name.split("/")[0])

#  Tokenization (For Rule-Based Classifier)
vectorizer = CountVectorizer(
    stop_words="english",
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2)
)

def fast_tokenize(text):
    return set(vectorizer.build_analyzer()(text))

#  Length Penalty
def apply_length_penalty(score, doc_length):
    return score / (1 + math.log(1 + doc_length) / 70)

#  Rule-Based Classification
def classify_document(text):
    doc_words = fast_tokenize(text)
    doc_length = len(doc_words)

    if doc_length < 100:
        return None, 0.0  # Skip classification if document too short

    best_match, best_score = None, 0
    for topic in main_topics:
        topic_keywords = set(word for subtopic in top_keywords_per_topic if subtopic.startswith(topic) for word in top_keywords_per_topic[subtopic])
        matched_words = doc_words.intersection(topic_keywords)

        weighted_score = sum(1.0 * (0.85 ** idx) for idx, word in enumerate(matched_words))
        max_possible_score = sum(1.0 * (0.85 ** idx) for idx in range(len(topic_keywords)))
        normalized_score = weighted_score / max_possible_score if max_possible_score > 0 else 0

        adjusted_score = apply_length_penalty(normalized_score, doc_length)

        if adjusted_score > best_score and len(matched_words) >= 30:
            best_score = adjusted_score
            best_match = topic

    if best_score < 0.9:
        return None, best_score  

    return best_match, best_score

#  LLM
def evaluate_topic_with_llama(file_content):
    """Classify the text using AWS Bedrock (Meta's Llama 3.3 70B Instruct)."""
    try:
        prompt = f"""
        Analyze the following document and classify it into only one of these topics: {unique_topics_str}.
        After explaining your reasoning, clearly state the final topic at the end.

        Document:
        {file_content}

        Explanation:
        Final Topic:
        """

        formatted_prompt = f"""
        <|begin_of_text|>
        <|start_header_id|>user<|end_header_id|>
        {prompt}
        <|eot_id|>
        <|start_header_id|>assistant<|end_header_id|>
        """

        response = bedrock_client.invoke_model(
            modelId=MODEL_ID_LLAMA,
            body=json.dumps({
                "prompt": formatted_prompt,
                "max_gen_len": 512,
                "temperature": 0,
            }),
            contentType="application/json"
        )

        response_body = json.loads(response['body'].read())
        response_text = response_body.get("generation", "").strip()

        match = re.search(r"Final Topic:\s*(.+)", response_text, re.IGNORECASE)
        predicted_topic = match.group(1).strip() if match else "Unknown"

        return predicted_topic, response_text

    except Exception as e:
        print(f"Error calling AWS Bedrock API: {e}")
        return "Unknown", "Error occurred during LLM call"

#  Main Pipeline (Single File Upload)
def process_single_file():
    with open("processed_file.pkl", "rb") as f:
        metadata = pickle.load(f)

    file_path = metadata["file_path"]
    filename = metadata["filename"]

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Rule-Based Attempt
    predicted_topic, confidence = classify_document(content)

    if predicted_topic:
        print(f" Rule-Based Classification: {predicted_topic} (Confidence: {confidence:.2f})")
        return filename, predicted_topic, confidence, "-", "Rule-Based"

    # Fallback to LLM (full document)
    predicted_topic, explanation = evaluate_topic_with_llama(content)

    print(f"LLM Classification: {predicted_topic}")
    return filename, predicted_topic, "-", explanation, "LLM"

#  Run and Output Results
if __name__ == "__main__":
    filename, predicted_topic, confidence, explanation, source = process_single_file()

    result = {
        "Filename": filename,
        "Predicted Topic": predicted_topic,
        "Confidence": confidence,
        "Explanation": explanation,
        "Source": source
    }
    print(json.dumps(result, indent=4))


LLM Classification: Annual Reports
{
    "Filename": "gp-financial-1q-2018",
    "Predicted Topic": "Annual Reports",
    "Confidence": "-",
    "Explanation": "The document provided appears to be a financial report of United Overseas Bank Limited, detailing its unaudited financial results for the first quarter ended March. The report includes information on the bank's financial performance, such as net interest income, non-interest income, operating expenses, and allowance for expected credit losses. It also provides an analysis of the bank's business segments, geographical segments, and capital adequacy ratios.\n\nThe report is written in a formal and technical tone, using financial terminology and jargon, which suggests that it is intended for an audience with a background in finance or banking. The level of detail and the specific information provided, such as the bank's financial statements and capital adequacy ratios, also suggest that the report is intended for regulatory or inv

# Demo with Rule Based Classification and Sample

In [21]:
import os
import re
import io
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
from nltk.corpus import stopwords
import pickle

def clean_text(text):
    """
    Clean text by removing unwanted characters and formatting.
    """
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII characters
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters and numbers
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    text = re.sub(r'\n+', '\n', text)  # Remove extra newlines
    return text


def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)


def preprocess_pdf(file_content, filename):
    """Extract and preprocess text from uploaded PDF file."""
    try:
        # Convert memoryview to BytesIO
        pdf_stream = io.BytesIO(file_content)

        reader = PdfReader(pdf_stream)
        extracted_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                extracted_text += text + "\n"

        cleaned_text = clean_text(extracted_text)
        cleaned_text = remove_stop_words(cleaned_text)

        # Save cleaned text to file
        output_path = os.path.join(os.getcwd(), f"{filename}.txt")
        with open(output_path, "w", encoding="utf-8") as text_file:
            text_file.write(cleaned_text)

        print(f"Processed text saved to: {output_path}")
        print(f"Preview:\n{cleaned_text[:500]}...")  # Print first 500 chars

        # Save metadata for second script
        metadata = {
            "filename": filename,
            "file_path": output_path
        }
        with open("processed_file.pkl", "wb") as f:
            pickle.dump(metadata, f)

        print("Metadata saved for evaluation script.")

    except Exception as e:
        print(f"Error processing PDF: {e}")


# Upload Widget
upload_widget = widgets.FileUpload(
    accept='.pdf',  # Accept only PDF files
    multiple=False  # Only allow single file upload
)


def on_upload_change(change):
    """Handle file upload and process PDF."""
    if upload_widget.value:
        for file_info in upload_widget.value:
            filename = file_info['name'].replace(".pdf", "")
            file_content = file_info['content'].tobytes()  # Convert memoryview to bytes
            
            # Process PDF
            preprocess_pdf(file_content, filename)


# Attach event listener
upload_widget.observe(on_upload_change, names='value')

display(upload_widget)

FileUpload(value=(), accept='.pdf', description='Upload')

In [22]:
import os
import pickle
import json
import random
import pandas as pd
import math
import re
from sklearn.feature_extraction.text import CountVectorizer
import boto3
from botocore.config import Config
from dotenv import load_dotenv

#  Load Environment Variables
load_dotenv("codes.env")

aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
aws_region = os.environ.get("AWS_REGION")

MODEL_ID_LLAMA = "us.meta.llama3-3-70b-instruct-v1:0"
config = Config(read_timeout=1000)

bedrock_client = boto3.client(
    "bedrock-runtime",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    config=config
)

#  Load Predefined Topics & Mappings
mapping_file_path = 'final_file_topic_mapping.csv'
file_topic_mapping = pd.read_csv(mapping_file_path)
unique_topics = file_topic_mapping['folder_name'].unique().tolist()
unique_topics_str = ', '.join(unique_topics)

#  Load Keyword Data for Rule-Based Classification
top_keywords_per_topic = {}
main_topics = set()

df = pd.read_csv("refined_tfidf_bigrams.csv")
for index, row in df.iterrows():
    topic_name = row.iloc[0].strip()
    keywords = [row[col] for col in df.columns[1:] if pd.notna(row[col])]
    if keywords:
        top_keywords_per_topic[topic_name] = keywords[:150]
        main_topics.add(topic_name.split("/")[0])

#  Tokenization (For Rule-Based Classifier)
vectorizer = CountVectorizer(
    stop_words="english",
    lowercase=True,
    token_pattern=r"(?u)\b\w+\b",
    ngram_range=(1, 2)
)

def fast_tokenize(text):
    return set(vectorizer.build_analyzer()(text))

#  Length Penalty
def apply_length_penalty(score, doc_length):
    return score / (1 + math.log(1 + doc_length) / 70)

#  Rule-Based Classification
def classify_document(text):
    doc_words = fast_tokenize(text)
    doc_length = len(doc_words)

    if doc_length < 100:
        return None, 0.0  # Skip classification if document too short

    best_match, best_score = None, 0
    for topic in main_topics:
        topic_keywords = set(word for subtopic in top_keywords_per_topic if subtopic.startswith(topic) for word in top_keywords_per_topic[subtopic])
        matched_words = doc_words.intersection(topic_keywords)

        weighted_score = sum(1.0 * (0.85 ** idx) for idx, word in enumerate(matched_words))
        max_possible_score = sum(1.0 * (0.85 ** idx) for idx in range(len(topic_keywords)))
        normalized_score = weighted_score / max_possible_score if max_possible_score > 0 else 0

        adjusted_score = apply_length_penalty(normalized_score, doc_length)

        if adjusted_score > best_score and len(matched_words) >= 30:
            best_score = adjusted_score
            best_match = topic

    if best_score < 0.9:
        return None, best_score  # Low confidence → No classification (fall back to LLM)

    return best_match, best_score

#  Sampling Logic for Hybrid Text Extraction (Intro, Middle Sample, Conclusion)
def extract_intro_middle_conclusion(text, max_tokens=20000):
    words = text.split()
    total_words = len(words)

    if total_words < 5000:
        ratios = (0.15, 0.15, 0.15)
    elif total_words < 20000:
        ratios = (0.08, 0.12, 0.08)
    elif total_words < 50000:
        ratios = (0.04, 0.08, 0.04)
    else:
        ratios = (0.02, 0.06, 0.02)

    intro_end = max(int(total_words * ratios[0]), 100)
    conclusion_start = max(int(total_words * (1 - ratios[2])), total_words - 100)

    middle = words[intro_end:conclusion_start]
    middle_sample_size = int(len(middle) * ratios[1])
    middle_sample = random.sample(middle, min(middle_sample_size, len(middle)))

    hybrid_text_words = words[:intro_end] + middle_sample + words[conclusion_start:]
    estimated_tokens = len(hybrid_text_words) * 1.3

    if estimated_tokens > max_tokens:
        allowed_words = int(max_tokens / 1.3)
        hybrid_text_words = hybrid_text_words[:allowed_words]

    return " ".join(hybrid_text_words)

#  LLM Fallback (Uses Sampled Text)
def evaluate_topic_with_llama(file_content):
    try:
        prompt = f"""
        Analyze the following document sample and classify it into only one of these topics: {unique_topics_str}.
        After explaining your reasoning, clearly state the final topic at the end.

        Document:
        {file_content}

        Explanation:
        Final Topic:
        """

        formatted_prompt = f"""
        <|begin_of_text|>
        <|start_header_id|>user<|end_header_id|>
        {prompt}
        <|eot_id|>
        <|start_header_id|>assistant<|end_header_id|>
        """

        response = bedrock_client.invoke_model(
            modelId=MODEL_ID_LLAMA,
            body=json.dumps({
                "prompt": formatted_prompt,
                "max_gen_len": 512,
                "temperature": 0,
            }),
            contentType="application/json"
        )

        response_body = json.loads(response['body'].read())
        response_text = response_body.get("generation", "").strip()

        match = re.search(r"Final Topic:\s*(.+)", response_text, re.IGNORECASE)
        predicted_topic = match.group(1).strip() if match else "Unknown"

        return predicted_topic, response_text

    except Exception as e:
        print(f"Error calling AWS Bedrock API: {e}")
        return "Unknown", "Error occurred during LLM call"

#  Main Pipeline (Single File Upload)
def process_single_file():
    with open("processed_file.pkl", "rb") as f:
        metadata = pickle.load(f)

    file_path = metadata["file_path"]
    filename = metadata["filename"]

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Rule-Based Attempt
    predicted_topic, confidence = classify_document(content)

    if predicted_topic:
        print(f" Rule-Based Classification: {predicted_topic} (Confidence: {confidence:.2f})")
        return filename, predicted_topic, confidence, "-", "Rule-Based"

    # Fallback to LLM using Sampled Text (Hybrid)
    sampled_text = extract_intro_middle_conclusion(content)
    predicted_topic, explanation = evaluate_topic_with_llama(sampled_text)

    print(f" LLM Classification: {predicted_topic}")
    return filename, predicted_topic, "-", explanation, "LLM"

#  Run and Output Results
if __name__ == "__main__":
    filename, predicted_topic, confidence, explanation, source = process_single_file()

    result = {
        "Filename": filename,
        "Predicted Topic": predicted_topic,
        "Confidence": confidence,
        "Explanation": explanation,
        "Source": source
    }
    print(json.dumps(result, indent=4))


 Rule-Based Classification: Annual Reports (Confidence: 0.90)
{
    "Filename": "selected-financial-statements-2021-en",
    "Predicted Topic": "Annual Reports",
    "Confidence": 0.9012374092614694,
    "Explanation": "-",
    "Source": "Rule-Based"
}


# Replaced Rule-Based Classification with Trainable Random Forest Model

In [5]:
import os
import re
import io
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
from nltk.corpus import stopwords
import pickle

def clean_text(text):
    """
    Clean text by removing unwanted characters and formatting.
    """
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII characters
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters and numbers
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    text = re.sub(r'\n+', '\n', text)  # Remove extra newlines
    return text


def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)


def preprocess_pdf(file_content, filename):
    """Extract and preprocess text from uploaded PDF file."""
    try:
        # Convert memoryview to BytesIO
        pdf_stream = io.BytesIO(file_content)

        reader = PdfReader(pdf_stream)
        extracted_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                extracted_text += text + "\n"

        cleaned_text = clean_text(extracted_text)
        cleaned_text = remove_stop_words(cleaned_text)

        # Save cleaned text to file
        output_path = os.path.join(os.getcwd(), f"{filename}.txt")
        with open(output_path, "w", encoding="utf-8") as text_file:
            text_file.write(cleaned_text)

        print(f"Processed text saved to: {output_path}")
        print(f"Preview:\n{cleaned_text[:500]}...")  # Print first 500 chars

        # Save metadata for second script
        metadata = {
            "filename": filename,
            "file_path": output_path
        }
        with open("processed_file.pkl", "wb") as f:
            pickle.dump(metadata, f)

        print("Metadata saved for evaluation script.")

    except Exception as e:
        print(f"Error processing PDF: {e}")


# Upload Widget
upload_widget = widgets.FileUpload(
    accept='.pdf',  # Accept only PDF files
    multiple=False  # Only allow single file upload
)


def on_upload_change(change):
    """Handle file upload and process PDF."""
    if upload_widget.value:
        for file_info in upload_widget.value:
            filename = file_info['name'].replace(".pdf", "")
            file_content = file_info['content'].tobytes()  # Convert memoryview to bytes
            
            # Process PDF
            preprocess_pdf(file_content, filename)


# Attach event listener
upload_widget.observe(on_upload_change, names='value')

display(upload_widget)

FileUpload(value=(), accept='.pdf', description='Upload')

In [ ]:
import os
import pickle
import json
import random
import pandas as pd
import math
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
import boto3
from botocore.config import Config
from dotenv import load_dotenv

#  Load Environment Variables
load_dotenv("codes.env")

aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
aws_region = os.environ.get("AWS_REGION")

MODEL_ID_LLAMA = "us.meta.llama3-3-70b-instruct-v1:0"
config = Config(read_timeout=1000)

bedrock_client = boto3.client(
    "bedrock-runtime",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    config=config
)

#  Load Predefined Topics & Mappings
mapping_file_path = 'final_file_topic_mapping.csv'
file_topic_mapping = pd.read_csv(mapping_file_path)
unique_topics = file_topic_mapping['folder_name'].unique().tolist()
unique_topics_str = ', '.join(unique_topics)

#  Load Trained Model & Vectorizer** 
print(" Loading Trained TF-IDF Vectorizer and Random Forest Model...")
tfidf_vectorizer = joblib.load("tfidf_vectorizer.pkl")
rf_model = joblib.load("rf_model.pkl")

#  Random Forest Classification** 
def rf_classify_document(text, confidence_threshold=0.99):
    """Classifies a document using the trained Random Forest model."""
    text_tfidf = tfidf_vectorizer.transform([text])  # Convert text to TF-IDF features
    y_pred_proba = rf_model.predict_proba(text_tfidf)[0]  # Get probability scores
    max_prob = max(y_pred_proba)
    
    if max_prob < confidence_threshold:
        return None, max_prob  # Defer to LLM

    predicted_topic = rf_model.classes_[y_pred_proba.argmax()]
    return predicted_topic, max_prob

#  Sampling Logic for Hybrid Text Extraction**
def extract_intro_middle_conclusion(text, max_tokens=20000):
    words = text.split()
    total_words = len(words)

    if total_words < 5000:
        ratios = (0.15, 0.15, 0.15)
    elif total_words < 20000:
        ratios = (0.08, 0.12, 0.08)
    elif total_words < 50000:
        ratios = (0.04, 0.08, 0.04)
    else:
        ratios = (0.02, 0.06, 0.02)

    intro_end = max(int(total_words * ratios[0]), 100)
    conclusion_start = max(int(total_words * (1 - ratios[2])), total_words - 100)

    middle = words[intro_end:conclusion_start]
    middle_sample_size = int(len(middle) * ratios[1])
    middle_sample = random.sample(middle, min(middle_sample_size, len(middle)))

    hybrid_text_words = words[:intro_end] + middle_sample + words[conclusion_start:]
    estimated_tokens = len(hybrid_text_words) * 1.3

    if estimated_tokens > max_tokens:
        allowed_words = int(max_tokens / 1.3)
        hybrid_text_words = hybrid_text_words[:allowed_words]

    return " ".join(hybrid_text_words)

#  LLM Fallback (Uses Sampled Text)**
def evaluate_topic_with_llama(file_content):
    try:
        prompt = f"""
        Analyze the following document sample and classify it into only one of these topics: {unique_topics_str}.
        After explaining your reasoning, clearly state the final topic at the end.

        Document:
        {file_content}

        Explanation:
        Final Topic:
        """

        formatted_prompt = f"""
        <|begin_of_text|>
        <|start_header_id|>user<|end_header_id|>
        {prompt}
        <|eot_id|>
        <|start_header_id|>assistant<|end_header_id|>
        """

        response = bedrock_client.invoke_model(
            modelId=MODEL_ID_LLAMA,
            body=json.dumps({
                "prompt": formatted_prompt,
                "max_gen_len": 512,
                "temperature": 0,
            }),
            contentType="application/json"
        )

        response_body = json.loads(response['body'].read())
        response_text = response_body.get("generation", "").strip()

        match = re.search(r"Final Topic:\s*(.+)", response_text, re.IGNORECASE)
        predicted_topic = match.group(1).strip() if match else "Unknown"

        return predicted_topic, response_text

    except Exception as e:
        print(f"Error calling AWS Bedrock API: {e}")
        return "Unknown", "Error occurred during LLM call"

#  Main Pipeline (Single File Upload)**
def process_single_file():
    with open("processed_file.pkl", "rb") as f:
        metadata = pickle.load(f)

    file_path = metadata["file_path"]
    filename = metadata["filename"]

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    sampled_text = extract_intro_middle_conclusion(content)
    predicted_topic, confidence = rf_classify_document(sampled_text)

    if predicted_topic:
        print(f" Random Forest Classification: {predicted_topic} (Confidence: {confidence:.2f})")
        return filename, predicted_topic, confidence, "-", "Random Forest Rule-Based Classification"

    # Fallback to LLM using Sampled Text (Hybrid)
    predicted_topic, explanation = evaluate_topic_with_llama(sampled_text)

    print(f" LLM Classification: {predicted_topic}")
    return filename, predicted_topic, "-", explanation, "LLM"

#  Run and Output Results**
if __name__ == "__main__":
    filename, predicted_topic, confidence, explanation, source = process_single_file()

    result = {
        "Filename": filename,
        "Predicted Topic": predicted_topic,
        "Confidence": confidence,
        "Explanation": explanation,
        "Source": source
    }
    print(json.dumps(result, indent=4))


 Loading Trained TF-IDF Vectorizer and Random Forest Model...
✅ Max Prob: 1.0000, Predicted Topic: Risk Management
✅ Top 5 Probabilities: [1.0, 0.0, 0.0, 0.0, 0.0]
 Random Forest Classification: Risk Management (Confidence: 1.00)
{
    "Filename": "pillar3-disclosures-2q-2017",
    "Predicted Topic": "Risk Management",
    "Confidence": 1.0,
    "Explanation": "-",
    "Source": "Random Forest Rule-Based Classification"
}


# Final Rule-Based Classification with Trainable Random Forest
Under this section, we defer all topics predicted as "Financial Regulations" to the Large Language Model, since precision for Financial Regulations is significantly lower than the rest.

In [7]:
import os
import re
import io
import ipywidgets as widgets
from IPython.display import display
from pypdf import PdfReader
from nltk.corpus import stopwords
import pickle

def clean_text(text):
    """
    Clean text by removing unwanted characters and formatting.
    """
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII characters
    text = re.sub(r'http\S+', '', text)  # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove special characters and numbers
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra whitespace
    text = re.sub(r'\n+', '\n', text)  # Remove extra newlines
    return text


def remove_stop_words(text):
    stop_words = set(stopwords.words('english'))
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return ' '.join(filtered_words)


def preprocess_pdf(file_content, filename):
    """Extract and preprocess text from uploaded PDF file."""
    try:
        # Convert memoryview to BytesIO
        pdf_stream = io.BytesIO(file_content)

        reader = PdfReader(pdf_stream)
        extracted_text = ""
        for page in reader.pages:
            text = page.extract_text()
            if text:
                extracted_text += text + "\n"

        cleaned_text = clean_text(extracted_text)
        cleaned_text = remove_stop_words(cleaned_text)

        # Save cleaned text to file
        output_path = os.path.join(os.getcwd(), f"{filename}.txt")
        with open(output_path, "w", encoding="utf-8") as text_file:
            text_file.write(cleaned_text)

        print(f"Processed text saved to: {output_path}")
        print(f"Preview:\n{cleaned_text[:500]}...")  # Print first 500 chars

        # Save metadata for second script
        metadata = {
            "filename": filename,
            "file_path": output_path
        }
        with open("processed_file.pkl", "wb") as f:
            pickle.dump(metadata, f)

        print("Metadata saved for evaluation script.")

    except Exception as e:
        print(f"Error processing PDF: {e}")


# Upload Widget
upload_widget = widgets.FileUpload(
    accept='.pdf',  # Accept only PDF files
    multiple=False  # Only allow single file upload
)


def on_upload_change(change):
    """Handle file upload and process PDF."""
    if upload_widget.value:
        for file_info in upload_widget.value:
            filename = file_info['name'].replace(".pdf", "")
            file_content = file_info['content'].tobytes()  # Convert memoryview to bytes
            
            # Process PDF
            preprocess_pdf(file_content, filename)


# Attach event listener
upload_widget.observe(on_upload_change, names='value')

display(upload_widget)

FileUpload(value=(), accept='.pdf', description='Upload')

In [10]:
import os
import pickle
import json
import random
import pandas as pd
import math
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
import boto3
from botocore.config import Config
from dotenv import load_dotenv

#  Load Environment Variables
load_dotenv("codes.env")

aws_access_key = os.environ.get("AWS_ACCESS_KEY_ID")
aws_secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")
aws_region = os.environ.get("AWS_REGION")

MODEL_ID_LLAMA = "us.meta.llama3-3-70b-instruct-v1:0"
config = Config(read_timeout=1000)

bedrock_client = boto3.client(
    "bedrock-runtime",
    region_name=aws_region,
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    config=config
)

#  Load Predefined Topics & Mappings
mapping_file_path = 'final_file_topic_mapping.csv'
file_topic_mapping = pd.read_csv(mapping_file_path)
unique_topics = file_topic_mapping['folder_name'].unique().tolist()
unique_topics_str = ', '.join(unique_topics)

#  Load Trained Model & Vectorizer** 
print(" Loading Trained TF-IDF Vectorizer and Random Forest Model...")
tfidf_vectorizer = joblib.load("tfidf_vectorizer.pkl")
rf_model = joblib.load("rf_model.pkl")

#  Random Forest Classification** 
def rf_classify_document(text, confidence_threshold=0.99):
    """Classifies a document using the trained Random Forest model."""
    text_tfidf = tfidf_vectorizer.transform([text])  # Convert text to TF-IDF features
    y_pred_proba = rf_model.predict_proba(text_tfidf)[0]  # Get probability scores
    
    max_prob = max(y_pred_proba)
    predicted_topic = rf_model.classes_[y_pred_proba.argmax()]

    if max_prob < confidence_threshold:
        return None, max_prob  # Defer to LLM

    return predicted_topic, max_prob


#  Sampling Logic for Hybrid Text Extraction**
def extract_intro_middle_conclusion(text, max_tokens=20000):
    words = text.split()
    total_words = len(words)

    if total_words < 5000:
        ratios = (0.15, 0.15, 0.15)
    elif total_words < 20000:
        ratios = (0.08, 0.12, 0.08)
    elif total_words < 50000:
        ratios = (0.04, 0.08, 0.04)
    else:
        ratios = (0.02, 0.06, 0.02)

    intro_end = max(int(total_words * ratios[0]), 100)
    conclusion_start = max(int(total_words * (1 - ratios[2])), total_words - 100)

    middle = words[intro_end:conclusion_start]
    middle_sample_size = int(len(middle) * ratios[1])
    middle_sample = random.sample(middle, min(middle_sample_size, len(middle)))

    hybrid_text_words = words[:intro_end] + middle_sample + words[conclusion_start:]
    estimated_tokens = len(hybrid_text_words) * 1.3

    if estimated_tokens > max_tokens:
        allowed_words = int(max_tokens / 1.3)
        hybrid_text_words = hybrid_text_words[:allowed_words]

    return " ".join(hybrid_text_words)

#  LLM Fallback (Uses Sampled Text)**
def evaluate_topic_with_llama(file_content):
    try:
        prompt = f"""
        Analyze the following document sample and classify it into only one of these topics: {unique_topics_str}.
        After explaining your reasoning, clearly state the final topic at the end.

        Document:
        {file_content}

        Explanation: <Your explanation>
        Final Topic: <One of the topics from the list>
        """

        formatted_prompt = f"""
        <|begin_of_text|>
        <|start_header_id|>user<|end_header_id|>
        {prompt}
        <|eot_id|>
        <|start_header_id|>assistant<|end_header_id|>
        """

        response = bedrock_client.invoke_model(
            modelId=MODEL_ID_LLAMA,
            body=json.dumps({
                "prompt": formatted_prompt,
                "max_gen_len": 512,
                "temperature": 0,
            }),
            contentType="application/json"
        )

        response_body = json.loads(response['body'].read())
        response_text = response_body.get("generation", "").strip()

        match = re.search(r"Final Topic:\s*(.+)", response_text, re.IGNORECASE)
        predicted_topic = match.group(1).strip() if match else "Unknown"

        return predicted_topic, response_text

    except Exception as e:
        print(f"Error calling AWS Bedrock API: {e}")
        return "Unknown", "Error occurred during LLM call"

#  Main Pipeline (Single File Upload)**
def process_single_file():
    with open("processed_file.pkl", "rb") as f:
        metadata = pickle.load(f)

    file_path = metadata["file_path"]
    filename = metadata["filename"]

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    sampled_text = extract_intro_middle_conclusion(content)
    predicted_topic, confidence = rf_classify_document(sampled_text)

    if predicted_topic:
        print(f" Random Forest Classification: {predicted_topic} (Confidence: {confidence:.2f})")
        return filename, predicted_topic, confidence, "-", "Random Forest Rule-Based Classification"

    # Fallback to LLM using Sampled Text (Hybrid)
    predicted_topic, explanation = evaluate_topic_with_llama(sampled_text)

    print(f" LLM Classification: {predicted_topic}")
    return filename, predicted_topic, "-", explanation, "LLM"

#  Run and Output Results**
if __name__ == "__main__":
    filename, predicted_topic, confidence, explanation, source = process_single_file()

    result = {
        "Filename": filename,
        "Predicted Topic": predicted_topic,
        "Confidence": confidence,
        "Explanation": explanation,
        "Source": source
    }
    print(json.dumps(result, indent=4))

 Loading Trained TF-IDF Vectorizer and Random Forest Model...
 Random Forest Classification: Anti Money Laundering (Confidence: 1.00)
{
    "Filename": "Guidance for Effective AML CFT Transaction Monitoring Controls",
    "Predicted Topic": "Anti Money Laundering",
    "Confidence": 1.0,
    "Explanation": "-",
    "Source": "Random Forest Rule-Based Classification"
}
